# Person 2 — Step 3: Thumbnail Analysis

Three-tier approach:

| Tier | Tool | What it measures | Runs on |
|------|------|------------------|---------|
| 1 | **OpenCV** | Brightness, contrast, colorfulness, saturation, text-overlay pixel ratio | All thumbnails |
| 2 | **CLIP** (zero-shot) | Semantic content: face present, text-heavy, animated, diagram, etc. | All thumbnails |
| 3 | **Claude Vision API** | Rich description + appeal/professionalism score | Stratified sample ~300 |

## Install deps (run once)
```bash
pip install opencv-python-headless pillow torch torchvision open_clip_torch tqdm anthropic
```

In [ ]:
import sys
sys.path.append('../../')

import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from tqdm.notebook import tqdm

from shared.config import PERSON2_DIR, THUMBNAILS_DIR

FIGURES = PERSON2_DIR / 'outputs' / 'figures'
TABLES  = PERSON2_DIR / 'outputs' / 'tables'
MANIFEST = PERSON2_DIR / 'outputs' / 'thumbnail_manifest.csv'

manifest = pd.read_csv(MANIFEST)
# Only work with successfully downloaded thumbnails
manifest = manifest[manifest['download_status'] == 'ok'].copy()
print(f"Working with {len(manifest)} thumbnails")

---
## TIER 1 — OpenCV low-level visual features

In [ ]:
def opencv_features(path: str) -> dict | None:
    """
    Returns low-level visual features for a thumbnail.
    colorfulness: Hasler & Süsstrunk (2003) metric
    """
    img = cv2.imread(path)
    if img is None:
        return None

    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(float)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(float)

    R, G, B = rgb[:,:,0].astype(float), rgb[:,:,1].astype(float), rgb[:,:,2].astype(float)
    rg = R - G
    yb = 0.5 * (R + G) - B
    colorfulness = np.sqrt(rg.std()**2 + yb.std()**2) + 0.3 * np.sqrt(rg.mean()**2 + yb.mean()**2)

    # Brightness (perceived luminance)
    brightness = 0.299*R.mean() + 0.587*G.mean() + 0.114*B.mean()

    # Contrast (Michelson)
    contrast = (gray.max() - gray.min()) / (gray.max() + gray.min() + 1e-6)

    # Saturation
    saturation = hsv[:,:,1].mean()

    # Edge density — proxy for how visually busy/complex the thumbnail is
    edges = cv2.Canny(img, 100, 200)
    edge_density = edges.mean() / 255.0

    # Dominant hue (circular mean)
    hues = hsv[:,:,0].flatten()
    dominant_hue = np.degrees(np.arctan2(
        np.sin(np.radians(hues * 2)).mean(),
        np.cos(np.radians(hues * 2)).mean()
    )) / 2 % 180

    return {
        'brightness':    brightness,
        'contrast':      contrast,
        'saturation':    saturation,
        'colorfulness':  colorfulness,
        'edge_density':  edge_density,
        'dominant_hue':  dominant_hue,
    }

print("Testing on one thumbnail...")
sample_path = manifest['thumb_path'].iloc[0]
opencv_features(sample_path)

In [ ]:
cv_results = []
for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc='OpenCV features'):
    feats = opencv_features(row['thumb_path'])
    if feats:
        feats['video_id'] = row['video_id']
        cv_results.append(feats)

cv_df = pd.DataFrame(cv_results)
cv_df.to_csv(TABLES / 'opencv_features.csv', index=False)
print(f"Saved {len(cv_df)} rows")
cv_df.describe()

In [ ]:
# Merge with manifest for group label
cv_merged = cv_df.merge(manifest[['video_id', 'group_label', 'view_count', 'like_count', 'comment_count']], on='video_id')

PALETTE = {'independent': '#2ecc71', 'institutional': '#3498db'}
visual_feats = ['brightness', 'contrast', 'saturation', 'colorfulness', 'edge_density']

fig, axes = plt.subplots(1, len(visual_feats), figsize=(18, 4))
for ax, feat in zip(axes, visual_feats):
    for grp, color in PALETTE.items():
        vals = cv_merged[cv_merged['group_label'].str.lower() == grp][feat].dropna()
        vals.hist(ax=ax, bins=40, alpha=0.6, color=color, label=grp.title(), density=True)
    ax.set_title(feat.replace('_', ' ').title())
    ax.legend(fontsize=7)
plt.suptitle('Thumbnail visual features: Independent vs. Institutional')
plt.tight_layout()
plt.savefig(FIGURES / 'opencv_distributions.png', dpi=150)
plt.show()

---
## TIER 2 — CLIP zero-shot semantic classification

CLIP lets us ask visual questions in plain English without any training data.
We define pairs of opposing text prompts and score each thumbnail against them.

In [ ]:
import torch
import open_clip
from PIL import Image

# Load CLIP — ViT-B/32 is fast; upgrade to ViT-L/14 for better accuracy
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
tokenizer = open_clip.get_tokenizer('ViT-B-32')
model.eval()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(DEVICE)
print(f"CLIP on {DEVICE}")

In [ ]:
# Each question is a (positive_label, list_of_prompts) pair.
# The score = softmax probability of the first prompt in the list.
CLIP_QUESTIONS = {
    'has_face': [
        'a YouTube thumbnail showing a human face or person',
        'a YouTube thumbnail with no people, only text, objects or graphics'
    ],
    'is_animated': [
        'an animated or illustrated YouTube thumbnail, cartoon style',
        'a photographic, real-world YouTube thumbnail'
    ],
    'has_text_overlay': [
        'a YouTube thumbnail with large bold text or title overlay',
        'a YouTube thumbnail with no text or very minimal text'
    ],
    'is_diagram': [
        'a scientific diagram, graph, equation, or whiteboard content',
        'a lifestyle, interview, or entertainment style thumbnail'
    ],
    'is_colorful': [
        'a very bright and colorful YouTube thumbnail with vivid colors',
        'a dark, muted, or monochrome YouTube thumbnail'
    ],
    'is_professional': [
        'a polished, professionally designed YouTube thumbnail',
        'a simple, low-effort or raw screenshot thumbnail'
    ],
    'is_clickbait': [
        'a clickbait, sensational, shocking or exaggerated YouTube thumbnail',
        'a calm, factual, educational YouTube thumbnail'
    ],
}

# Pre-encode all text prompts
encoded_questions = {}
for key, prompts in CLIP_QUESTIONS.items():
    tokens = tokenizer(prompts).to(DEVICE)
    with torch.no_grad():
        text_feats = model.encode_text(tokens)
        text_feats /= text_feats.norm(dim=-1, keepdim=True)
    encoded_questions[key] = text_feats

print("Text prompts encoded")

In [ ]:
BATCH_SIZE = 64

def load_image(path):
    try:
        return preprocess(Image.open(path).convert('RGB'))
    except Exception:
        return None

all_clip_rows = []

paths = manifest['thumb_path'].tolist()
video_ids = manifest['video_id'].tolist()

for start in tqdm(range(0, len(paths), BATCH_SIZE), desc='CLIP batches'):
    batch_paths = paths[start:start+BATCH_SIZE]
    batch_ids   = video_ids[start:start+BATCH_SIZE]

    imgs, valid_ids = [], []
    for pid, vid in zip(batch_paths, batch_ids):
        img = load_image(pid)
        if img is not None:
            imgs.append(img)
            valid_ids.append(vid)

    if not imgs:
        continue

    batch_tensor = torch.stack(imgs).to(DEVICE)
    with torch.no_grad():
        img_feats = model.encode_image(batch_tensor)
        img_feats /= img_feats.norm(dim=-1, keepdim=True)

    for i, vid in enumerate(valid_ids):
        row = {'video_id': vid}
        for key, text_feats in encoded_questions.items():
            # cosine similarities → softmax probability for first prompt (positive class)
            sim = (img_feats[i] @ text_feats.T) * 100
            prob = torch.softmax(sim, dim=0)[0].item()
            row[f'clip_{key}'] = prob
        all_clip_rows.append(row)

clip_df = pd.DataFrame(all_clip_rows)
clip_df.to_csv(TABLES / 'clip_features.csv', index=False)
print(f"Saved CLIP features for {len(clip_df)} thumbnails")
clip_df.head()

In [ ]:
clip_merged = clip_df.merge(manifest[['video_id', 'group_label']], on='video_id')

clip_cols = [c for c in clip_df.columns if c.startswith('clip_')]
grp_means = clip_merged.groupby('group_label')[clip_cols].mean().T

fig, ax = plt.subplots(figsize=(10, 5))
grp_means.plot(kind='barh', ax=ax, color=['#2ecc71', '#3498db'])
ax.set_xlabel('Mean CLIP probability (0–1)')
ax.set_title('CLIP zero-shot classification: Independent vs. Institutional')
ax.axvline(0.5, color='grey', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.savefig(FIGURES / 'clip_classification_comparison.png', dpi=150)
plt.show()
grp_means.round(3)

---
## TIER 3 — Claude Vision API (stratified sample)

We can't run Claude on all 9,500 thumbnails (cost + latency), so we take
a **stratified sample of ~300** thumbnails (proportional per channel) and get
rich structured descriptions back.

Set your API key: `export ANTHROPIC_API_KEY=sk-ant-...`

In [ ]:
import anthropic
import base64
import json
import os
import time

client = anthropic.Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY'))

SAMPLE_PER_CHANNEL = 40  # 8 channels × 40 = 320 total

sample = (
    manifest
    .groupby('channel_title', group_keys=False)
    .apply(lambda g: g.sample(min(len(g), SAMPLE_PER_CHANNEL), random_state=42))
    .reset_index(drop=True)
)
print(f"Sample size: {len(sample)} thumbnails across {sample['channel_title'].nunique()} channels")

In [ ]:
VISION_PROMPT = """Analyse this YouTube thumbnail and respond with a JSON object containing ONLY these fields:

{
  "has_face": true/false,
  "face_count": integer (0 if none),
  "emotion": "excited" | "serious" | "neutral" | "confused" | "none",
  "has_text_overlay": true/false,
  "text_size": "large" | "medium" | "small" | "none",
  "visual_style": "animation" | "real_photo" | "diagram" | "screen_recording" | "mixed",
  "color_scheme": "bright_vivid" | "dark_moody" | "neutral_muted" | "high_contrast" | "monochrome",
  "complexity": 1-5 (1=very simple, 5=very busy/complex),
  "professionalism": 1-5 (1=amateurish, 5=highly polished),
  "appeal_score": 1-5 (1=unlikely to get clicks, 5=very eye-catching),
  "curiosity_trigger": true/false (does it make you want to watch?),
  "main_subject": brief string describing the primary subject/content,
  "channel_type_guess": "independent_creator" | "institutional"
}

Return ONLY valid JSON, no explanation."""

def analyse_thumbnail(path: str) -> dict | None:
    try:
        img_bytes = Path(path).read_bytes()
        b64 = base64.standard_b64encode(img_bytes).decode()
        response = client.messages.create(
            model='claude-sonnet-4-6',
            max_tokens=400,
            messages=[{
                'role': 'user',
                'content': [
                    {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/jpeg', 'data': b64}},
                    {'type': 'text', 'text': VISION_PROMPT}
                ]
            }]
        )
        text = response.content[0].text.strip()
        # Strip markdown code blocks if present
        if text.startswith('```'):
            text = text.split('```')[1]
            if text.startswith('json'):
                text = text[4:]
        return json.loads(text)
    except Exception as e:
        print(f"Error on {path}: {e}")
        return None

In [ ]:
# Run on sample — adds a small sleep to stay within API rate limits
vision_results = []

for _, row in tqdm(sample.iterrows(), total=len(sample), desc='Claude Vision'):
    result = analyse_thumbnail(row['thumb_path'])
    if result:
        result['video_id'] = row['video_id']
        result['channel_title'] = row['channel_title']
        result['group_label'] = row['group_label']
        vision_results.append(result)
    time.sleep(0.3)  # ~3 req/s — well within Anthropic limits

vision_df = pd.DataFrame(vision_results)
vision_df.to_csv(TABLES / 'claude_vision_features.csv', index=False)
print(f"Saved {len(vision_df)} Claude vision annotations")
vision_df.head()

In [ ]:
# --- Summary visualisation ---
numeric_vision = ['complexity', 'professionalism', 'appeal_score']
bool_vision    = ['has_face', 'has_text_overlay', 'curiosity_trigger']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Numeric scores
score_means = vision_df.groupby('group_label')[numeric_vision].mean().T
score_means.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#3498db'], rot=0)
axes[0].set_ylim(1, 5)
axes[0].set_title('Claude scores (1–5): Independent vs. Institutional')
axes[0].set_ylabel('Mean score')

# Boolean features
bool_means = vision_df.groupby('group_label')[bool_vision].mean().mul(100).T
bool_means.plot(kind='barh', ax=axes[1], color=['#2ecc71', '#3498db'])
axes[1].set_xlabel('% of thumbnails')
axes[1].set_title('Binary thumbnail features')

plt.tight_layout()
plt.savefig(FIGURES / 'claude_vision_summary.png', dpi=150)
plt.show()

In [ ]:
# Confusion matrix: did Claude guess the channel type correctly?
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

true_labels  = vision_df['group_label'].str.lower()
pred_labels  = vision_df['channel_type_guess'].str.replace('independent_creator', 'independent')

print(classification_report(true_labels, pred_labels))

cm = confusion_matrix(true_labels, pred_labels, labels=['independent', 'institutional'])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['independent', 'institutional'],
            yticklabels=['independent', 'institutional'], cmap='Blues', ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Can Claude guess the channel type from the thumbnail?')
plt.tight_layout()
plt.savefig(FIGURES / 'claude_type_guess_cm.png', dpi=150)
plt.show()

In [ ]:
# Visual style breakdown
style_counts = (
    vision_df.groupby(['group_label', 'visual_style'])
    .size()
    .reset_index(name='count')
    .pivot(index='visual_style', columns='group_label', values='count')
    .fillna(0)
)
# Normalise to percentages
style_pct = style_counts.div(style_counts.sum()) * 100

style_pct.plot(kind='bar', figsize=(8, 4), color=['#2ecc71', '#3498db'], rot=30)
plt.ylabel('% of thumbnails')
plt.title('Visual style breakdown by channel type')
plt.tight_layout()
plt.savefig(FIGURES / 'visual_style_breakdown.png', dpi=150)
plt.show()
style_pct.round(1)